# Try RO-Crate metadata validation with a Python dictionary

Set `INPUT_PATH` in the first setup cell to the JSON file you want to inspect. The notebook loads it into `INPUT` and validates that same dictionary with `packaging_mode` set to `auto`, `attached`, and `detached`.

The first setup cell also contains the profile, offline-mode, and optional Attached crate directory parameters. Run this notebook from the repository checkout with its `.venv` kernel selected. Dictionary-only input has no payload backend, so validation is metadata-only. Full Attached validation is available when you also set `ATTACHED_CRATE_DIR` as the `package_root`.

This notebook is also published in the documentation. Download it to run the examples locally; building the documentation does not execute validations.

In [ ]:
import json
from pathlib import Path

from rocrate_validator.models.settings import ValidationSettings
from rocrate_validator.services import validate
from rocrate_validator.utils.io_helpers.output.console import Console
from rocrate_validator.utils.io_helpers.output.text.layout.report import ValidationReportLayout

# Parameters: edit these values to try another metadata file or profile.
INPUT_PATH = Path("/tmp/mymetadata.json")
PROFILE_IDENTIFIER = "ro-crate-1.3"
OFFLINE = False
ATTACHED_CRATE_DIR = Path("/tmp/profile-crate-process-run-crate/")  # Set this to a local crate directory

with INPUT_PATH.open(encoding="utf-8") as stream:
    INPUT = json.load(stream)

if not isinstance(INPUT, dict):
    raise TypeError(f"Expected a JSON object, got {type(INPUT).__name__}")

print(f"Loaded {INPUT_PATH}: {len(INPUT.get('@graph', []))} graph entities")
print("Top-level keys:", list(INPUT))


## Run one validation

Use the parameters in the first setup cell to change the input file, target profile, or offline behavior. `OFFLINE=False` allows uncached JSON-LD contexts or external resources to be fetched; set it to `True` to use the validator cache only.

Each run displays the same report header used by the CLI, followed by a JSON summary. With `package_root`, the `RO-Crate` row shows the supplied directory; dictionary-only input shows `in-memory metadata`.


In [ ]:
def run_validation(package_mode="auto", metadata_only=True, package_root=None):
    settings = ValidationSettings(
        rocrate_uri=None,  # Not needed for dictionary input
        metadata_dict=INPUT,
        package_root=package_root,
        profile_identifier=PROFILE_IDENTIFIER,
        packaging_mode=package_mode,
        metadata_only=metadata_only,
        offline=OFFLINE,
    )
    result = validate(settings)
    report = result.to_dict()
    report_settings = report["validation_settings"]
    summary = {
        "requested_mode": package_mode,
        "resolved_package_type": report_settings["package_type"],
        "metadata_source": report_settings["metadata_source"],
        "metadata_only": report_settings["metadata_only"],
        "package_root": report_settings.get("package_root"),
        "passed": report["passed"],
        "issues": len(report["issues"]),
        "skipped_checks": report["skipped_checks"],
    }
    console = Console(force_jupyter=True)
    layout = ValidationReportLayout(console, result.validation_settings)
    layout.result = result
    _ = layout.layout
    if layout.base_info_layout is not None:
        console.print(layout.base_info_layout.renderable)
    print(json.dumps(summary, indent=2))
    return result, report

result, report = run_validation(package_mode="auto", metadata_only=True)


## Compare package modes

Each run uses the same `INPUT` dictionary. `auto` resolves dictionary input to `UNSPECIFIED`; `attached` and `detached` select their respective profile rules. The report includes the resolved package type, validation scope, issues, and skipped checks.


In [ ]:
mode_results = {}
for mode in ("auto", "attached", "detached"):
    print(f"\n=== packaging_mode={mode} ===")
    mode_results[mode] = run_validation(package_mode=mode, metadata_only=True)


## Explore the `metadata_only` option

A dictionary has no associated payload files. For `auto` and `detached`, the validator therefore makes the dictionary validation metadata-only even if `metadata_only=False` was requested. For explicit `attached`, `metadata_only=False` is rejected: provide `package_root` pointing to a local crate directory to perform full Attached validation with the dictionary.


In [ ]:
scope_results = {}
for mode in ("auto", "detached"):
    print(f"\n=== packaging_mode={mode}, metadata_only=False ===")
    scope_results[(mode, False)] = run_validation(package_mode=mode, metadata_only=False)

try:
    run_validation(package_mode="attached", metadata_only=False)
except ValueError as exc:
    print("Expected configuration error for Attached dictionary input:")
    print(exc)


## Dictionary plus attached directory with full validation

This combination is supported: the dictionary supplies the metadata graph and `ATTACHED_CRATE_DIR` supplies the payload files. The dictionary replaces the metadata descriptor: `ro-crate-metadata.json` is not required on disk and is ignored if present. Metadata checks use `INPUT` in both validation scopes. The report records `metadata_source="dictionary"` and the package root.

Set `ATTACHED_CRATE_DIR` in the first setup cell to an existing local RO-Crate directory before running this section. `auto` resolves this input to Attached; the cell runs both `auto` and explicit `attached`, each with metadata-only and full validation. In every case, the `RO-Crate` row should show `ATTACHED_CRATE_DIR`. Results are saved in `package_results`, indexed by `(mode, metadata_only)`.


In [ ]:
package_results = {}
if not ATTACHED_CRATE_DIR.is_dir():
    print(f"Set ATTACHED_CRATE_DIR to an existing crate directory first: {ATTACHED_CRATE_DIR}")
else:
    for mode in ("auto", "attached"):
        for metadata_only in (True, False):
            print(f"\n=== dictionary + package_root, packaging_mode={mode}, metadata_only={metadata_only} ===")
            package_results[(mode, metadata_only)] = run_validation(
                package_mode=mode,
                metadata_only=metadata_only,
                package_root=ATTACHED_CRATE_DIR,
            )

    print("\nDetached mode with package_root is an invalid combination:")
    try:
        run_validation(
            package_mode="detached",
            metadata_only=False,
            package_root=ATTACHED_CRATE_DIR,
        )
    except ValueError as exc:
        print(exc)


## Inspect findings

Print issue summaries for one mode. Select `auto`, `attached`, or `detached` below.

Set `INSPECT_PACKAGE_ROOT=True` to inspect a dictionary-plus-directory result from the preceding section; `SCOPE_TO_INSPECT` selects metadata-only (`True`) or full validation (`False`).


In [ ]:
MODE_TO_INSPECT = "attached"
INSPECT_PACKAGE_ROOT = False
SCOPE_TO_INSPECT = False
if INSPECT_PACKAGE_ROOT:
    key = (MODE_TO_INSPECT, SCOPE_TO_INSPECT)
    if key not in package_results:
        raise ValueError("Run the dictionary-plus-directory section first; select auto or attached.")
    selected_result, selected_report = package_results[key]
else:
    selected_result, selected_report = mode_results[MODE_TO_INSPECT]
print(f"\n=== inspecting mode: {MODE_TO_INSPECT} ===\n")
for issue in selected_report["issues"]:
    check = issue.get("check", {})
    print(f"[{issue['severity']}] {check.get('identifier', '(unknown check)')}: {issue['message']}")

print(f"\nSkipped checks: {selected_report['skipped_checks']}")
for skipped in selected_report.get("skipped_check_details", []):
    print(skipped)
